# VC Dimension

Wiki reference for [VC dimension](https://ml-viz-ruby.vercel.app/wiki/vc-dimension).

**The idea in one sentence.** The VC dimension is the largest number of points a hypothesis class
can **shatter** (realize every $\pm$ labelling of) — for a line in 2-D it is exactly **3** (3
points can be shattered, 4 cannot because of XOR) — and it drives the **generalization bound**:
the train/test gap grows with VC dimension and shrinks like $1/\sqrt{n}$.

We test shattering and the VC bound from scratch, **validate that a line shatters 3 but not 4
points and that the bound behaves correctly**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import itertools
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({
    'axes.facecolor': '#1a1d27',
    'figure.facecolor': '#0f1117',
    'axes.edgecolor': '#3a3d4a',
    'grid.color': '#2a2d3a',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'font.size': 11,
})
TEAL, ROSE, BRAND = '#14b8a6', '#f43f5e', '#6366f1'

## 1 · A linear-separability test

A labelling of points is realisable by a 2D linear classifier iff the $+$ and $-$ classes are **linearly separable**. We test this by sweeping the separating line's normal direction $\mathbf{w} = (\cos\theta, \sin\theta)$: project all points onto $\mathbf{w}$; the classes are separable along this direction if their projections don't overlap. If *any* direction works, the labelling is realisable.

In [ ]:
def linearly_separable(points, labels, n_angles=360):
    """True if a straight line realises this +/- labelling of `points`."""
    points = np.asarray(points, float)
    labels = np.asarray(labels, bool)
    if labels.all() or (~labels).all():
        return True  # one class empty: trivially separable
    for theta in np.linspace(0, np.pi, n_angles, endpoint=False):
        w = np.array([np.cos(theta), np.sin(theta)])
        proj = points @ w
        pos, neg = proj[labels], proj[~labels]
        if pos.max() < neg.min() or neg.max() < pos.min():
            return True
    return False

def fraction_shattered(points):
    """How many of the 2^m labellings a line can realise."""
    m = len(points)
    sep = sum(linearly_separable(points, lab)
              for lab in itertools.product([False, True], repeat=m))
    return sep, 2 ** m

# sanity check: 3 points in a triangle
tri = [(0.0, 0.0), (1.0, 1.7), (2.0, 0.0)]
print('triangle (3 pts):', fraction_shattered(tri), '→ shattered!' )

## 2 · Three points are shattered, four are not

We enumerate every labelling of a 3-point triangle (all 8 separable) and of a 4-point square (the diagonal **XOR** labelling fails).

In [ ]:
triangle = np.array([(0.0, 0.0), (1.0, 1.7), (2.0, 0.0)])
square   = np.array([(0.0, 0.0), (2.0, 0.0), (2.0, 2.0), (0.0, 2.0)])

for name, pts in [('3 points (triangle)', triangle), ('4 points (square)', square)]:
    sep, total = fraction_shattered(pts)
    verdict = 'SHATTERED' if sep == total else 'NOT shattered'
    print(f'{name:24s}  {sep:2d} / {total:2d} labellings separable  → {verdict}')

### Validate: a line shatters 3 points but not 4

A linear classifier can realize **all** $2^3 = 8$ labellings of 3 points in general position
(shattered), but **not all** $2^4 = 16$ labellings of 4 points — the XOR (diagonal) labelling is
impossible. So the VC dimension of a 2-D line is exactly 3. We confirm.

In [ ]:
sep3, tot3 = fraction_shattered(triangle)
sep4, tot4 = fraction_shattered(square)
print(f'3 points: {sep3}/{tot3} labellings separable;  4 points: {sep4}/{tot4}')
assert sep3 == tot3 == 8, '3 points in general position ARE shattered (all 8 labellings) -> VC dim >= 3'
assert sep4 < tot4, '4 points are NOT shattered (XOR fails) -> VC dim of a 2-D line is exactly 3'
print('\n✅ VC dimension = the most points you can shatter; for a line it is 3')

In [ ]:
# Visualise the one labelling of the square that no line can separate (XOR).
xor = np.array([True, False, True, False])  # diagonal points share a label
fig, ax = plt.subplots(figsize=(5, 5))
for p, lab in zip(square, xor):
    ax.scatter(*p, s=400, color=TEAL if lab else ROSE, edgecolor='#0f1117', zorder=3)
    ax.text(*p, '+' if lab else '−', ha='center', va='center', color='#0f1117', fontsize=18, fontweight='bold', zorder=4)
ax.set(title=f'XOR labelling of 4 points — separable? {linearly_separable(square, xor)}',
       xlim=(-1, 3), ylim=(-1, 3))
ax.set_aspect('equal'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Three points: 8 / 8 — every labelling realisable ⇒ $d_{\text{VC}} \ge 3$.  
Four points: at most 14 / 16 — the two XOR labellings always fail ⇒ no 4-set is shattered ⇒ $d_{\text{VC}} < 4$.

Together: **the VC dimension of a 2D linear classifier is exactly 3.**

## 3 · The growth function: exponential → polynomial

The growth function $\Pi(m)$ is the most distinct labellings a line can produce on $m$ points. It hugs the ceiling $2^m$ while $m \le d_{\text{VC}}$, then **Sauer's lemma** caps it at $\sum_{i=0}^{d}\binom{m}{i}$. We estimate $\Pi(m)$ empirically by counting separable labellings on random point clouds.

In [ ]:
from math import comb
rng = np.random.default_rng(0)

def growth_estimate(m, trials=40):
    """Max separable labellings over random m-point sets (estimate of Pi(m))."""
    best = 0
    for _ in range(trials):
        pts = rng.uniform(0, 1, (m, 2))
        sep = sum(linearly_separable(pts, lab)
                  for lab in itertools.product([False, True], repeat=m))
        best = max(best, sep)
    return best

d = 3  # VC dimension of 2D linear classifier
ms = range(1, 9)
pi_emp = [growth_estimate(m) for m in ms]
sauer  = [sum(comb(m, i) for i in range(d + 1)) for m in ms]
ceiling = [2 ** m for m in ms]

for m, e, s, c in zip(ms, pi_emp, sauer, ceiling):
    print(f'm={m}:  Pi(m)≈{e:4d}   Sauer bound={s:4d}   2^m={c:4d}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(list(ms), ceiling, '--', color='#64748b', label='$2^m$ (full shattering)')
ax.plot(list(ms), sauer, '-o', color=BRAND, label=r"Sauer bound $\sum_{i \leq d} C(m,i)$")
ax.plot(list(ms), pi_emp, '-o', color=TEAL, label='empirical $\\Pi(m)$')
ax.axvline(d, color=ROSE, ls=':', label=f'$d_{{VC}}={d}$')
ax.set(xlabel='number of points $m$', ylabel='distinct labellings', yscale='log',
       title='Growth function flattens past the VC dimension')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Up to $m = 3$ the empirical curve sits on $2^m$ (full shattering). From $m = 4$ it peels away and tracks the polynomial Sauer bound — the phase change that makes generalization possible.

## 4 · The VC generalization bound

The slogan is **gap $\sim \sqrt{d_{\text{VC}}/n}$**. The full confidence bound (probability $\ge 1-\delta$):

$$R(h) \le \hat{R}(h) + \sqrt{\tfrac{d_{\text{VC}}(\ln\frac{2n}{d_{\text{VC}}}+1) + \ln\frac{4}{\delta}}{n}}$$

We plot the confidence term against sample size $n$ for a few capacities.

In [ ]:
def vc_gap(n, dvc, delta=0.05):
    return np.sqrt((dvc * (np.log(2 * n / dvc) + 1) + np.log(4 / delta)) / n)

n = np.arange(50, 20001)
fig, ax = plt.subplots(figsize=(7, 4.5))
for dvc, c in [(3, TEAL), (50, BRAND), (500, ROSE)]:
    ax.plot(n, vc_gap(n, dvc), color=c, label=f'$d_{{VC}}={dvc}$')
ax.set(xlabel='training size $n$', ylabel='generalization-gap bound', xscale='log',
       title='More capacity loosens the bound; more data tightens it')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Validate: the generalization bound behaves correctly

The VC generalization gap **shrinks** as the training set grows and **grows** with the VC
dimension (model capacity). We confirm both monotonicities.

In [ ]:
print(f'gap at n=50 -> {vc_gap(50, 3):.3f};  at n=20000 -> {vc_gap(20000, 3):.3f}')
print(f'gap (dvc=3) -> {vc_gap(1000, 3):.3f};  (dvc=500) -> {vc_gap(1000, 500):.3f}')
assert vc_gap(20000, 3) < vc_gap(50, 3), 'the generalization gap shrinks as training size grows'
assert vc_gap(1000, 500) > vc_gap(1000, 3), 'higher VC dimension (more capacity) means a larger gap'
print('\n✅ more data closes the gap; more capacity widens it — the bias-capacity trade-off')

Every curve falls like $\sim 1/\sqrt{n}$, and higher $d_{\text{VC}}$ shifts the whole curve up: a higher-capacity class needs roughly proportionally more data to reach the same guaranteed gap. That is the entire moral of VC theory in one picture.

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **high VC dimension** | more capacity -> larger gap, overfitting risk (verified) |
| **loose bounds** | VC bounds are worst-case and often very loose in practice |
| **diminishing data returns** | gap $\sim 1/\sqrt n$ (demo) — 4x data to halve it |
| **infinite VC dim** | some classes (e.g. 1-NN) can't be bounded this way |
| **capacity != params** | VC dim can differ wildly from the parameter count |

Demo: quadrupling the data roughly halves the generalization bound.

In [ ]:
# The practical consequence of the VC bound: the gap shrinks like 1/sqrt(n), so returns to more
# data DIMINISH. To HALVE the generalization gap you must QUADRUPLE the training set — the same
# reason doubling your data gives less improvement than the first batch did. We confirm the
# quarter-for-half scaling.
g1 = vc_gap(1000, dvc=10)
g4 = vc_gap(4000, dvc=10)
print(f'gap at n=1000 -> {g1:.3f};  at n=4000 (4x data) -> {g4:.3f};  ratio {g4/g1:.2f}')
assert abs(g4 / g1 - 0.5) < 0.1, 'quadrupling the data roughly halves the bound (gap ~ 1/sqrt(n))'
print('\nGap ~ 1/sqrt(n) -> 4x the data to halve the gap: diminishing returns on more data.')

## ✏️ Your turn

Three short exercises. Each has a `# TODO(you)` blank, an `assert` that passes silently when you're right, and a hidden solution.

---

### Exercise 1 — VC dimension of an axis-aligned interval (1D)

A classifier on the line predicts $+$ inside an interval $[a, b]$ and $-$ outside. **What is its VC dimension?** Verify by brute force: write `interval_separable(points, labels)` — a labelling is realisable iff all the $+$ points form a contiguous block when sorted by position (no $-$ point sits between two $+$ points).

In [ ]:
def interval_separable(points, labels):
    """True if the + points are contiguous when sorted along the line."""
    points = np.asarray(points, float).ravel()
    labels = np.asarray(labels, bool)
    order = np.argsort(points)
    sorted_labels = labels[order]
    # TODO(you): return True iff the True entries form a single contiguous run.
    # Hint: count how many times the label flips False<->True along sorted_labels;
    #       a contiguous block of + flips at most twice.
    flips = ...
    return flips <= 2

def frac_interval(points):
    m = len(points)
    sep = sum(interval_separable(points, lab)
              for lab in itertools.product([False, True], repeat=m))
    return sep, 2 ** m

# 2 points shatter (4/4); 3 points do NOT (the +,-,+ labelling fails) → VC = 2
assert frac_interval([0.0, 1.0]) == (4, 4)
assert frac_interval([0.0, 1.0, 2.0])[0] == 7   # 8 - 1 failing labelling
print('Exercise 1 passed — the interval classifier has VC dimension 2.')

<details><summary>Solution</summary>

```python
flips = int(np.sum(sorted_labels[1:] != sorted_labels[:-1]))
```

A single contiguous block of `+` toggles the label at most twice (False→True entering the block, True→False leaving it). The `+,-,+` labelling on 3 sorted points flips 3 times, so it fails — hence 3 points can't be shattered and the VC dimension is **2**.

</details>

### Exercise 2 — Confirm 4 points are never shattered

Using `fraction_shattered` from Part 2, draw **many** random 4-point clouds and confirm that *none* of them is fully shattered (i.e. `sep` is always strictly less than 16). Fill in the blank.

In [ ]:
rng2 = np.random.default_rng(7)
max_sep = 0
for _ in range(300):
    pts = rng2.uniform(0, 1, (4, 2))
    # TODO(you): get the number of separable labellings for this 4-point cloud
    sep, total = ...
    max_sep = max(max_sep, sep)

assert max_sep < 16, 'a 4-point set should never reach all 16 labellings'
print(f'Best seen over 300 random 4-point sets: {max_sep}/16 — never shattered. ✓')

<details><summary>Solution</summary>

```python
sep, total = fraction_shattered(pts)
```

The most a generic 4-point set reaches is 14/16 — the two complementary XOR labellings are always unrealisable, which is the upper-bound half of the proof that $d_{\text{VC}} = 3$.

</details>

### Exercise 3 — How much data to halve the gap?

The VC gap scales like $\sqrt{d_{\text{VC}}/n}$. Ignoring the log factor, **by what factor must $n$ grow to cut the gap in half?** Set `factor` to the right number and check it numerically with `vc_gap`.

In [ ]:
# TODO(you): gap ~ 1/sqrt(n). To halve the gap, n must grow by what factor?
factor = ...

g1 = vc_gap(1000, dvc=10)
g2 = vc_gap(1000 * factor, dvc=10)
assert abs(g2 / g1 - 0.5) < 0.1, f'expected ~half the gap, got ratio {g2 / g1:.2f}'
print(f'n × {factor} → gap ratio {g2 / g1:.2f} (≈ 0.5). Quadrupling data roughly halves the bound.')

<details><summary>Solution</summary>

```python
factor = 4
```

Since the gap behaves like $1/\sqrt{n}$, halving it requires $\sqrt{n}$ to double, i.e. $n \to 4n$. (The log term makes the real ratio slightly above 0.5, which is why the test allows a tolerance.) Diminishing returns on data are baked into the $\sqrt{\cdot}$.

</details>

## Key takeaways

- **Shattering:** a class shatters points if it can realize *every* labelling; VC dim is the max
  shatterable count.
- **A 2-D line has VC dim 3** — shatters 3, not 4 (XOR) (verified).
- **The VC bound:** the gap grows with VC dimension and shrinks with $n$ (verified).
- **Diminishing returns:** gap $\sim 1/\sqrt{n}$, so 4× the data halves the bound (demo).